In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/README.md
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/18184.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19090.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/18177.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/16773.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19830.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/16786.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19090.hea
/kag

In [49]:
# ============================================================
# Notebook 02
# Dataset Generation for SCD Prediction
# ============================================================

"""
Purpose
-------
Generate ECG datasets following the paper.

Input
-----
MIT-BIH Sudden Cardiac Death Database
MIT-BIH Normal Sinus Rhythm Database

Output
------
30 min
25 min
20 min
15 min
10 min
5 min

Each contains

Positive (Pre-VF)
Negative (Normal Sinus)

Author:
Rashid Shahariar

"""

'\nPurpose\n-------\nGenerate ECG datasets following the paper.\n\nInput\n-----\nMIT-BIH Sudden Cardiac Death Database\nMIT-BIH Normal Sinus Rhythm Database\n\nOutput\n------\n30 min\n25 min\n20 min\n15 min\n10 min\n5 min\n\nEach contains\n\nPositive (Pre-VF)\nNegative (Normal Sinus)\n\nAuthor:\nRashid Shahariar\n\n'

In [50]:
pip install wfdb

Note: you may need to restart the kernel to use updated packages.


In [51]:
import os
import random
import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

In [52]:
pip install wfdb

Note: you may need to restart the kernel to use updated packages.


In [53]:
# ============================================================
# Dataset Paths
# ============================================================

DATA_ROOT = "/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter"

SCD_PATH = os.path.join(
    DATA_ROOT,
    "sudden-cardiac-death-holter-database-1.0.0"
)

NSR_PATH = os.path.join(
    DATA_ROOT,
    "nsrdb_250hz"
)

print("SCD Path Exists :", os.path.exists(SCD_PATH))
print("NSR Path Exists :", os.path.exists(NSR_PATH))

SCD Path Exists : True
NSR Path Exists : True


In [54]:
usable_records = [
    "30","31","32","33","34",
    "35","36","37","38","39",
    "41","43","44","45","46",
    "47","48","50","51","52"
]

In [55]:
# ============================================================
# Automatically Extract SCD Metadata
# ============================================================

def extract_scd_metadata(scd_path, usable_records):

    metadata = []

    for record in usable_records:

        hea_file = os.path.join(scd_path, f"{record}.hea")

        recording_start = None
        vf_onset = None

        with open(hea_file, "r") as f:

            lines = f.readlines()

        # ---------- First line ----------
        # Example:
        # 30 2 250 22099250 12:00:00

        first_line = lines[0].split()

        recording_start = first_line[-1]

        # ---------- Remaining lines ----------
        for line in lines:

            if "#vfon:" in line:

                vf_onset = line.split(": ", 1)[1].strip()

                break

        metadata.append({

            "Record": record,
            "Recording_Start": recording_start,
            "VF_Onset": vf_onset

        })

    return pd.DataFrame(metadata)

In [56]:
scd_metadata = extract_scd_metadata(
    SCD_PATH,
    usable_records
)

scd_metadata

,Record,Recording_Start,VF_Onset
0,30,12:00:00,07:54:33
1,31,10:15:00,13:42:24
2,32,5:04:00,16:45:18
3,33,10:38:00,04:46:19
4,34,12:00:00,06:35:44
5,35,12:00:00,24:34:56
6,36,15:09:00,18:59:01
7,37,14:29:00,01:31:13
8,38,12:00:00,08:01:54
9,39,12:00:00,04:37:51


In [57]:
##############         Helper function




def time_to_seconds(time_str):

    h, m, s = map(int, time_str.split(":"))

    return h*3600 + m*60 + s


def seconds_to_samples(seconds, fs):

    return int(seconds * fs)


def minutes_to_samples(minutes, fs):

    return int(minutes * 60 * fs)

In [58]:
print("="*60)

print(scd_metadata)

print("="*60)

   Record Recording_Start  VF_Onset
0      30        12:00:00  07:54:33
1      31        10:15:00  13:42:24
2      32         5:04:00  16:45:18
3      33        10:38:00  04:46:19
4      34        12:00:00  06:35:44
5      35        12:00:00  24:34:56
6      36        15:09:00  18:59:01
7      37        14:29:00  01:31:13
8      38        12:00:00  08:01:54
9      39        12:00:00  04:37:51
10     41        11:45:00  02:59:24
11     43        12:00:00  15:37:11
12     44        14:51:00  19:38:45
13     45        12:00:00  18:09:17
14     46        10:20:00  03:41:47
15     47        12:00:00  06:13:01
16     48        10:15:00  02:29:40
17     50        21:50:00  11:45:43
18     51        12:00:00  22:58:23
19     52        12:00:00  02:32:40


In [59]:
record_id = "30"

record = wfdb.rdrecord(
    os.path.join(SCD_PATH, record_id)
)

signal = record.p_signal
fs = record.fs

print("Record:", record_id)
print("Sampling rate:", fs)
print("Signal shape:", signal.shape)

Record: 30
Sampling rate: 250
Signal shape: (22099250, 2)


In [60]:
# ============================================================
# Extract Pre-VF Window
# ============================================================

def extract_pre_vf_window(record_path,
                          vf_time,
                          prediction_minutes,
                          fs=250):
    """
    Extract the ECG signal immediately before VF onset.

    Parameters
    ----------
    record_path : str
        Path to the WFDB record (without extension)

    vf_time : str
        VF onset time in HH:MM:SS format

    prediction_minutes : int
        Minutes before VF (30, 25, 20, 15, 10, or 5)

    fs : int
        Sampling frequency (default = 250 Hz)

    Returns
    -------
    numpy.ndarray
        ECG window of shape (samples, 2)
    """

    record = wfdb.rdrecord(record_path)
    signal = record.p_signal

    vf_sample = int(time_to_seconds(vf_time) * fs)

    start_sample = vf_sample - minutes_to_samples(prediction_minutes, fs)
    end_sample = vf_sample

    return signal[start_sample:end_sample]

In [61]:
record_id = "30"

record_path = os.path.join(SCD_PATH, record_id)

vf_time = scd_metadata.loc[
    scd_metadata["Record"] == record_id,
    "VF_Onset"
].iloc[0]

window30 = extract_pre_vf_window(
    record_path=record_path,
    vf_time=vf_time,
    prediction_minutes=30,
    fs=250
)

print(window30.shape)

(450000, 2)


In [62]:
# ============================================================
# Create Output Directory
# ============================================================

OUTPUT_DIR = "/kaggle/working/SCD_PreVF"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(OUTPUT_DIR)

/kaggle/working/SCD_PreVF


In [63]:
# ============================================================
# Extract and Save Immediately
# ============================================================

import gc
import numpy as np

prediction_windows = [30, 25, 20, 15, 10, 5]

for minute in prediction_windows:

    save_dir = os.path.join(
        OUTPUT_DIR,
        f"{minute}_min"
    )

    os.makedirs(save_dir, exist_ok=True)

    print(f"\n========== {minute} Minutes ==========")

    for _, row in scd_metadata.iterrows():

        record_id = row["Record"]
        vf_time = row["VF_Onset"]

        record_path = os.path.join(SCD_PATH, record_id)

        ecg = extract_pre_vf_window(
            record_path=record_path,
            vf_time=vf_time,
            prediction_minutes=minute,
            fs=250
        )

        save_path = os.path.join(
            save_dir,
            f"{record_id}.npy"
        )

        np.save(save_path, ecg)

        print(
            f"Saved Record {record_id} -> {ecg.shape}"
        )

        # Free memory
        del ecg
        gc.collect()


========== 30 Minutes ==========
Saved Record 30 -> (450000, 2)
Saved Record 31 -> (450000, 2)
Saved Record 32 -> (450000, 2)
Saved Record 33 -> (450000, 2)
Saved Record 34 -> (450000, 2)
Saved Record 35 -> (450000, 2)
Saved Record 36 -> (450000, 2)
Saved Record 37 -> (450000, 2)
Saved Record 38 -> (450000, 2)
Saved Record 39 -> (450000, 2)
Saved Record 41 -> (450000, 2)
Saved Record 43 -> (450000, 2)
Saved Record 44 -> (450000, 2)
Saved Record 45 -> (450000, 2)
Saved Record 46 -> (450000, 2)
Saved Record 47 -> (450000, 2)
Saved Record 48 -> (450000, 2)
Saved Record 50 -> (450000, 2)
Saved Record 51 -> (450000, 2)
Saved Record 52 -> (450000, 2)

========== 25 Minutes ==========
Saved Record 30 -> (375000, 2)
Saved Record 31 -> (375000, 2)
Saved Record 32 -> (375000, 2)
Saved Record 33 -> (375000, 2)
Saved Record 34 -> (375000, 2)
Saved Record 35 -> (375000, 2)
Saved Record 36 -> (375000, 2)
Saved Record 37 -> (375000, 2)
Saved Record 38 -> (375000, 2)
Saved Record 39 -> (375000, 2)
Sa

In [64]:
for minute in prediction_windows:

    folder = os.path.join(
        OUTPUT_DIR,
        f"{minute}_min"
    )

    print(f"\n{minute} Minutes")

    print(sorted(os.listdir(folder))[:5])

    print("Total Files:", len(os.listdir(folder)))


30 Minutes
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
Total Files: 20

25 Minutes
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
Total Files: 20

20 Minutes
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
Total Files: 20

15 Minutes
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
Total Files: 20

10 Minutes
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
Total Files: 20

5 Minutes
['30.npy', '31.npy', '32.npy', '33.npy', '34.npy']
Total Files: 20


In [65]:
sample = np.load(
    os.path.join(
        OUTPUT_DIR,
        "30_min",
        "30.npy"
    )
)

print(sample.shape)

(450000, 2)
